# Agents & Tool Use

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/building-with-llms/05-agents-and-tool-use

A from-scratch, runnable implementation of the concepts in the lesson — pure NumPy, no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A tiny ReAct agent

An agent calls **tools** and reasons over their results. We build a bounded ReAct loop with a scripted policy (standing in for the LLM) that emits `Action: tool[arg]` lines; our runtime executes the tool and feeds back an `Observation`.

In [ ]:
def calculator(expr):
    return str(eval(expr, {'__builtins__': {}}, {}))

FACTS = {'capital of france': 'Paris', 'population of paris': '2100000'}
def lookup(query):
    return FACTS.get(query.strip().lower(), 'unknown')

TOOLS = {'calculator': calculator, 'lookup': lookup}

# Scripted 'policy' — in a real agent the LLM produces these lines.
SCRIPT = [
    'Thought: I need the capital of France.\nAction: lookup[capital of france]',
    'Thought: Now its population.\nAction: lookup[population of paris]',
    'Thought: Divide by 1000.\nAction: calculator[2100000 / 1000]',
    'Thought: I have the answer.\nAnswer: about 2100',
]

In [ ]:
def run_agent(script, max_steps=6):
    for step in range(max_steps):
        msg = script[step]
        print(msg)
        m = re.search(r'Action:\s*(\w+)\[(.*?)\]', msg)
        if m:
            tool, arg = m.group(1), m.group(2)
            obs = TOOLS[tool](arg)
            print(f'Observation: {obs}\n')
        elif 'Answer:' in msg:
            print('\n[done]')
            return
    print('\n[stopped: step budget exhausted]')

run_agent(SCRIPT)

The `max_steps` budget is the **bounded loop** that stops a runaway agent. Note we sandbox `eval` (no builtins) — never feed unvalidated model output to a raw interpreter.

## ✏️ Your turn

Add a `length` tool that returns the number of characters in its argument, and register it in `TOOLS`.

In [ ]:
def length(s):
    # TODO(you): return the character count of s as a string.
    return ''

TOOLS['length'] = length
assert TOOLS['length']('paris') == '5'
print('passed ✓')

<details><summary>Solution</summary>

```python
def length(s):
    return str(len(s))
```

</details>